In [ ]:
import pandas as pd
import numpy as np

In [ ]:
MODEL_FILES = {
    "LightGBM": "Prescriptive/LightGBM_prescriptive_optimizer_inputs.csv",
    "RF": "Prescriptive/RF_prescriptive_optimizer_inputs.csv",
    "XGBoost": "Prescriptive/XGBoost_prescriptive_optimizer_inputs.csv",
}

In [ ]:
required_cols = [
    "LOG_ID",
    "DURATION_P10_MINS",
    "DURATION_P50_MINS",
    "DURATION_P90_MINS",
]

for model_name, path in MODEL_FILES.items():

    df = pd.read_csv(path)

    assert len(df) == 17083
    assert list(df.columns) == required_cols
    assert df["LOG_ID"].is_unique
    assert not df[required_cols].isna().any().any()

    assert np.all(
        df["DURATION_P10_MINS"]
        <= df["DURATION_P50_MINS"]
    )

    assert np.all(
        df["DURATION_P50_MINS"]
        <= df["DURATION_P90_MINS"]
    )

    print(model_name, "PASS", df.shape)

In [ ]:
lgb = pd.read_csv(MODEL_FILES["LightGBM"])
rf = pd.read_csv(MODEL_FILES["RF"])
xgb = pd.read_csv(MODEL_FILES["XGBoost"])

assert lgb["LOG_ID"].tolist() == rf["LOG_ID"].tolist()
assert lgb["LOG_ID"].tolist() == xgb["LOG_ID"].tolist()

print("Cross-model LOG_ID alignment: PASS")

In [ ]:
cohort = pd.read_csv("Cohort/fixed_cohort.csv")

for model_name, path in MODEL_FILES.items():
    pred = pd.read_csv(path)

    missing_ids = (
        set(cohort["LOG_ID"])
        - set(pred["LOG_ID"])
    )

    assert not missing_ids, (
        f"{model_name}: fixed cohort IDs missing: {missing_ids}"
    )

print("Fixed 12-case cohort compatibility: PASS")

In [ ]:
lambdas = np.round(
    np.arange(0.0, 1.01, 0.1),
    1
)

cap_rows = []

for model_name, path in MODEL_FILES.items():

    df = pd.read_csv(path)

    p50 = df["DURATION_P50_MINS"].to_numpy()
    p90 = df["DURATION_P90_MINS"].to_numpy()

    for lam in lambdas:

        uncapped = (
            p50
            + lam * (p90 - p50)
        )

        affected = uncapped > 360

        cap_rows.append({
            "Model": model_name,
            "Lambda": lam,
            "N": len(df),
            "N_Capped": int(affected.sum()),
            "Pct_Capped": float(
                affected.mean() * 100
            ),
            "Max_Uncapped": float(
                uncapped.max()
            )
        })

cap_audit = pd.DataFrame(cap_rows)

print(cap_audit)

In [ ]:
cap_audit.to_csv(
    "Results/duration_cap_audit.csv",
    index=False
)